# Notebooks and git

Everything else in session 5 happens in the terminal, so this is the one
notebook. It exists because notebooks and git get along badly, and knowing
why will save you a confusing afternoon.

## A notebook is not a text file

A `.py` file is exactly what you see. A `.ipynb` file is **JSON**: your code,
your prose, the output of every cell, and some bookkeeping, all wrapped up
together.

Here is what one actually looks like from the outside.

In [ ]:
import json
import os
from pathlib import Path

here = Path.cwd()
while not (here / "data").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

path = Path("sessions/session-02/notebooks/01-control-flow.ipynb")
raw = json.loads(path.read_text(encoding="utf-8"))

print("top-level keys:", list(raw.keys()))
print("cells:         ", len(raw["cells"]))
print()
print(json.dumps(raw["cells"][1], indent=2)[:400])

That is one cell. Note `"outputs": []` and `"execution_count": null`.

## Why that matters for git

Three consequences, in order of how much they will annoy you:

**1. Diffs are hard to read.** Change one word of a comment and git shows you
a JSON blob. `git diff` on a notebook is much less useful than on a script.

**2. Outputs get committed too.** Run a cell that prints a thousand rows and
that output is saved **inside the file**. Your two-kilobyte notebook becomes
two megabytes, and every run makes a new diff even when the code is
identical.

**3. Execution counts churn.** Those `[1]`, `[2]`, `[3]` numbers are stored.
Re-running the same notebook in a different order changes the file without
changing a single line of code.

## What to do about it

The short answer: **strip the outputs before you commit.**

This repo has a tool for it:

```
python3 tools/nbtool.py check     # are any notebooks carrying output?
python3 tools/nbtool.py strip     # remove it
```

`check` is worth running before you commit. It also catches a notebook whose
JSON has been damaged, which happens if two programs write it at once.

In [ ]:
outputs_saved = sum(1 for cell in raw["cells"] if cell.get("outputs"))
print(f"cells carrying saved output in that notebook: {outputs_saved}")

## The rule for this course

| Commit | Do not commit |
|---|---|
| your notebooks, with outputs stripped | notebooks full of saved output |
| your `.py` scripts | `.ipynb_checkpoints/` |
| your `.sql` files | `__pycache__/` |
| `MY-NOTES.md` | anything in `output/` |

`.ipynb_checkpoints/` is a folder Jupyter creates for its own autosaves. It
is already in this repo's `.gitignore`, along with `output/` and the rest.
Have a look:

In [ ]:
print(Path(".gitignore").read_text(encoding="utf-8"))

## One more habit worth having

A notebook can pass because of something you ran ten minutes ago and have
since deleted. Before you commit one, use **Restart and Run All**.

If it cannot run cleanly from top to bottom, it is not finished, however good
the output looks on screen. That is the notebook version of "does it actually
work", and it is the same check `tools/nbtool.py run` does for the whole
repo.

## Back to the terminal

The rest of this session is git itself. Everything you need is in
`sessions/session-05/demos/git_commands.md`, and the exercise is in
`sessions/session-05/exercises/README.md`.